<a href="https://colab.research.google.com/github/Rogerio-mack/IMT_CD_2026/blob/main/IMT_CD_Exercicio_pipeline_selecao_modelos_e_atributos_solucao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Exercício: Seleção de Modelos e Atributos**

Alunos: André Solano e Fernando Gimenez

# Exemplo: `Pipeline`

Veja mais em https://inria.github.io/scikit-learn-mooc/

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

# 1. Carregando o dataset Iris
iris = load_iris()
X, y = iris.data, iris.target

# Separação 20% dos dados para o teste final
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Criando o Pipeline estrutural com placeholders ('passthrough')
pipeline = Pipeline([
    ('scaler', 'passthrough'),
    ('classifier', 'passthrough')
])

# 3. Configurando a grade de busca com pré-processamento dinâmico
param_grid = [
    # Família 1: Árvore de Decisão (Não precisa de escala)
    {
        'scaler': ['passthrough'],
        'classifier': [DecisionTreeClassifier(random_state=42)],
        'classifier__max_depth': [3, 5, None],
        'classifier__criterion': ['gini', 'entropy']
    },
    # Família 2: SVC (Exige normalização)
    {
        'scaler': [StandardScaler()],
        'classifier': [SVC(random_state=42)],
        'classifier__C': [0.1, 1, 10],
        'classifier__kernel': ['linear', 'rbf']
    },
    # Família 3: Regressão Logística (Beneficia-se de escala)
    {
        'scaler': [StandardScaler()],
        'classifier': [LogisticRegression(random_state=42)],
        'classifier__C': [0.1, 1, 10],
        'classifier__solver': ['liblinear']
    }
]

# 4. Executando o GridSearchCV no conjunto de TREINO
grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

# Exibindo o melhor resultado encontrado na validação cruzada
print("--- FASE DE TREINAMENTO E VALIDAÇÃO ---")
print(f"Melhores parâmetros: {grid.best_params_}\n")

# 5. Aplicação do pipeline ao conjunto de teste

# O objeto 'grid' se comporta como o melhor modelo retreinado por padrão.
# Ele vai aplicar o scaler (se o campeão exigir) e fazer a predição automaticamente.
y_pred = grid.predict(X_test)

print("--- AVALIAÇÃO FINAL NO CONJUNTO DE TESTE ---")
print(f"Acurácia Final: {accuracy_score(y_test, y_pred):.4f}\n")
print("Relatório de Classificação Completo:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))


--- FASE DE TREINAMENTO E VALIDAÇÃO ---
Melhores parâmetros: {'classifier': SVC(random_state=42), 'classifier__C': 0.1, 'classifier__kernel': 'linear', 'scaler': StandardScaler()}

--- AVALIAÇÃO FINAL NO CONJUNTO DE TESTE ---
Acurácia Final: 0.9333

Relatório de Classificação Completo:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30



# Exercício 1

Empregue os conceitos de `pipeline` acima para fazer a classificação da espécie dos pinguins no dataset a seguir.

1. Você pode excluir dados ausentes
2. Empregue todas as variáveis preditores, fazendo o encode quando necessário
3. Para árvores de decisão e random forest, não empregue hot encode
4. Aplique o scale se necessário ou desejável
5. Verifique além dos modelos já empregados no exemplo, modelos de RandomForest e MLP com diferentes parâmetros (ao menos 3)

In [ ]:
df = sns.load_dataset("penguins")
df

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
import pandas as pd
import seaborn as sns

# 1. Carregando e limpando os dados (mantendo as colunas de texto/categóricas originais)
df_penguins = sns.load_dataset("penguins").dropna()

cat_cols = ["island", "sex"]
num_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

X_peng = df_penguins.drop(columns=["species"])
y_peng = df_penguins["species"]

# Divisão entre treino e teste
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_peng, y_peng, test_size=0.2, random_state=42, stratify=y_peng
)

# 2. Criando o Pipeline estrutural com placeholders ('passthrough')
pipeline = Pipeline([
    ('scaler', 'passthrough'),
    ('classifier', 'passthrough')
])

# Pré-processador para Árvores (Decision Tree e RandomForest):
# Usa OrdinalEncoder (equivalente ao Label Encoder para múltiplas colunas de features)
tree_preprocessor = ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols)
])

# Pré-processador para modelos lineares/MLP/SVM (exigem One-Hot e escala)
linear_preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

# 3. Configurando a grade de busca com pré-processamento dinâmico
param_grid = [
    # Família 1: Árvore de Decisão (com OrdinalEncoder, sem escala)
    {
        'scaler': [tree_preprocessor],
        'classifier': [DecisionTreeClassifier(random_state=42)],
        'classifier__max_depth': [3, 5, None],
        'classifier__criterion': ['gini', 'entropy']
    },
    # Família 2: Random Forest (com OrdinalEncoder, sem escala)
    {
        'scaler': [tree_preprocessor],
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [50, 100, 200],
        'classifier__max_depth': [3, 5, None]
    },
    # Família 3: SVC (Exige escala e One-Hot)
    {
        'scaler': [linear_preprocessor],
        'classifier': [SVC(random_state=42)],
        'classifier__C': [0.1, 1, 10],
        'classifier__kernel': ['linear', 'rbf']
    },
    # Família 4: Regressão Logística (Exige escala e One-Hot)
    {
        'scaler': [linear_preprocessor],
        'classifier': [LogisticRegression(random_state=42, max_iter=1000)],
        'classifier__C': [0.1, 1, 10],
        'classifier__solver': ['liblinear', 'lbfgs']
    },
    # Família 5: MLP (Exige escala e One-Hot)
    {
        'scaler': [linear_preprocessor],
        'classifier': [MLPClassifier(random_state=42, max_iter=1000)],
        'classifier__hidden_layer_sizes': [(50,), (100,), (50, 50)],
        'classifier__alpha': [0.0001, 0.001, 0.01]
    }
]

# 4. Executando o GridSearchCV
grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train_p, y_train_p)

# Resultados
print("--- FASE DE TREINAMENTO E VALIDAÇÃO ---")
print(f"Melhores parâmetros encontrados:\n{grid.best_params_}\n")

y_pred_p = grid.predict(X_test_p)

print("--- AVALIAÇÃO FINAL NO CONJUNTO DE TESTE ---")
print(f"Acurácia Final: {accuracy_score(y_test_p, y_pred_p):.4f}\n")
print("Relatório de Classificação Completo:")
print(classification_report(y_test_p, y_pred_p))

--- FASE DE TREINAMENTO E VALIDAÇÃO ---
Melhores parâmetros encontrados:
{'classifier': MLPClassifier(max_iter=1000, random_state=42), 'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (100,), 'scaler': ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['bill_length_mm', 'bill_depth_mm',
                                  'flipper_length_mm', 'body_mass_g']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['island', 'sex'])])}

--- AVALIAÇÃO FINAL NO CONJUNTO DE TESTE ---
Acurácia Final: 1.0000

Relatório de Classificação Completo:
              precision    recall  f1-score   support

      Adelie       1.00      1.00      1.00        29
   Chinstrap       1.00      1.00      1.00        14
      Gentoo       1.00      1.00      1.00        24

    accuracy                           1.00        67
   macro avg       1.00      1.00      1.00        67
weighted avg     

# Exercício 2

Verifique o ganho de informação do exercício anterior e refaça a seleção dos modelos empregando somente os 50% atributos de maior ganho.

**Qual a sua conclusão deste exercício?**

In [ ]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# Como o X_train_p original agora mantém strings, precisamos temporariamente usar
# um codificador rápido apenas para calcular o ganho de informação (mutual_info_classif exige dados numéricos)
X_train_encoded = X_train_p.copy()
for col in cat_cols:
    X_train_encoded[col] = X_train_encoded[col].astype('category').cat.codes

importances = mutual_info_classif(X_train_encoded, y_train_p, random_state=42)
feat_importances = pd.Series(importances, index=X_train_p.columns).sort_values(ascending=False)

print("--- GANHO DE INFORMAÇÃO DOS ATRIBUTOS ---")
print(feat_importances)

# Selecionando os 50% melhores atributos (3 melhores de 6)
top_k = len(feat_importances) // 2
selected_features = feat_importances.index[:top_k].tolist()
print(f"\nAtributos selecionados (Top 50%): {selected_features}\n")

# Filtrando os datasets de treino e teste mantendo as categorias textuais originais
X_train_p_filtered = X_train_p[selected_features]
X_test_p_filtered = X_test_p[selected_features]

# Separando as colunas numéricas e categóricas selecionadas para o novo pipeline
num_cols_sel = [c for c in selected_features if c in num_cols]
cat_cols_sel = [c for c in selected_features if c in cat_cols]

# Recriando os pré-processadores dinâmicos para os atributos selecionados
# Árvores: Usam OrdinalEncoder se houver variáveis categóricas entre as selecionadas
tree_preprocessor_sel = ColumnTransformer([
    ('num', 'passthrough', num_cols_sel),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols_sel)
])

# Modelos lineares/MLP/SVM: Usam OneHotEncoder e StandardScaler
linear_preprocessor_sel = ColumnTransformer([
    ('num', StandardScaler(), num_cols_sel),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols_sel)
])

# Ajustando a grade de parâmetros com a nova estrutura
param_grid_sel = [
    {
        'scaler': [tree_preprocessor_sel],
        'classifier': [DecisionTreeClassifier(random_state=42)],
        'classifier__max_depth': [3, 5, None],
        'classifier__criterion': ['gini', 'entropy']
    },
    {
        'scaler': [tree_preprocessor_sel],
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [50, 100, 200],
        'classifier__max_depth': [3, 5, None]
    },
    {
        'scaler': [linear_preprocessor_sel],
        'classifier': [SVC(random_state=42)],
        'classifier__C': [0.1, 1, 10],
        'classifier__kernel': ['linear', 'rbf']
    },
    {
        'scaler': [linear_preprocessor_sel],
        'classifier': [LogisticRegression(random_state=42, max_iter=1000)],
        'classifier__C': [0.1, 1, 10],
        'classifier__solver': ['liblinear', 'lbfgs']
    },
    {
        'scaler': [linear_preprocessor_sel],
        'classifier': [MLPClassifier(random_state=42, max_iter=1000)],
        'classifier__hidden_layer_sizes': [(50,), (100,), (50, 50)],
        'classifier__alpha': [0.0001, 0.001, 0.01]
    }
]

# Rodando a busca de modelos novamente usando o pipeline genérico instanciado no Ex 1
grid_sel = GridSearchCV(pipeline, param_grid_sel, cv=5, scoring='accuracy', n_jobs=-1)
grid_sel.fit(X_train_p_filtered, y_train_p)

print("--- NOVO TREINAMENTO (APENAS 50% MELHORES ATRIBUTOS) ---")
print(f"Melhores parâmetros encontrados:\n{grid_sel.best_params_}\n")

y_pred_sel = grid_sel.predict(X_test_p_filtered)

print("--- AVALIAÇÃO FINAL NO CONJUNTO DE TESTE FILTRADO ---")
print(f"Acurácia Final: {accuracy_score(y_test_p, y_pred_sel):.4f}\n")
print("Relatório de Classificação Completo:")
print(classification_report(y_test_p, y_pred_sel))

--- GANHO DE INFORMAÇÃO DOS ATRIBUTOS ---
flipper_length_mm    0.614498
bill_depth_mm        0.588360
bill_length_mm       0.564523
island               0.522546
body_mass_g          0.518304
sex                  0.000000
dtype: float64

Atributos selecionados (Top 50%): ['flipper_length_mm', 'bill_depth_mm', 'bill_length_mm']

--- NOVO TREINAMENTO (APENAS 50% MELHORES ATRIBUTOS) ---
Melhores parâmetros encontrados:
{'classifier': MLPClassifier(max_iter=1000, random_state=42), 'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (50, 50), 'scaler': ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['flipper_length_mm', 'bill_depth_mm',
                                  'bill_length_mm']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 [])])}

--- AVALIAÇÃO FINAL NO CONJUNTO DE TESTE FILTRADO ---
Acurácia Final: 1.0000

Relatório de Classificação Completo:
              p